[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yryo1005/OpenCampus_Demo/blob/main/OC_DepthAnything.ipynb)


# 深度推定デモ（Depth Anything）

1枚の写真から「手前と奥」の距離感を推定し，色で可視化するデモです．  
**Depth Anything V2**（Hugging Face Transformers）を使い，カメラやサンプル画像から深度マップを作ります．

**実行環境**: Google Colab（ランタイム → GPU: **T4** 推奨）

## セルの進め方
1. **設定**（Webカメラの左右反転・モデルサイズ・カラーマップなど）
2. **ライブラリのインストール**
3. **ライブラリの読み込み・モデル準備・サンプル画像のダウンロード**
4. **Gradio の起動**

> API キーは **不要** です（推論はすべて Colab 内で完結）．  
> Hugging Face のユーザー認証も **不要** です（公開モデルを自動ダウンロード）．  
> 初回はモデル（Small で約 100MB 前後）のダウンロードに時間がかかります．  
> インストール直後にエラーが出る場合は，**ランタイム → セッションを再起動**してから設定セルとセル3以降を再実行してください．


## 0. 設定

- カメラ映像が左右反転して見える場合は，次のセルの `MIRROR_WEBCAM` を切り替えてください（`True` = ミラー，`False` = 反転なし）．
- T4（VRAM 約 14GB）では既定の `small` を推奨します．`base` / `large` は精度は上がりますが重くなります．
- 変更後は **Gradio 起動セル**を再実行してください．


In [ ]:
# Webカメラの左右反転（ミラー表示）
# True  : 左右反転する（Gradio のデフォルトに近い自撮り表示）
# False : 左右反転しない
MIRROR_WEBCAM = True

# Depth Anything V2 のサイズ（小さいほど速い・軽い）
#   small : T4 向け推奨（約 25M パラメータ）
#   base  : より高精度・やや重い
#   large : 最高精度・重い（デモでは非推奨）
MODEL_SIZE = "small"

# 深度マップの色（OpenCV のカラーマップ名）
#   INFERNO / MAGMA / TURBO / PLASMA / VIRIDIS など
COLORMAP_NAME = "INFERNO"

# 推論前に長辺をこのピクセル以下へ縮小（VRAM・速度のバランス）
MAX_IMAGE_SIDE = 1024

print(f"MIRROR_WEBCAM = {MIRROR_WEBCAM}")
print(f"MODEL_SIZE = {MODEL_SIZE}")
print(f"COLORMAP_NAME = {COLORMAP_NAME}")
print(f"MAX_IMAGE_SIDE = {MAX_IMAGE_SIDE}")


## 1. ライブラリのインストール


In [ ]:
# Colab 標準の torch / transformers / gradio / opencv / Pillow を利用
# Depth Anything V2 対応のため transformers を念のため更新
!pip install -q -U "transformers>=4.45.0"


## 2. ライブラリの読み込み，変数のインスタンス化

サンプル画像をインターネットからダウンロードし，Depth Anything V2 を準備します．  
初回はモデルのダウンロードに数分かかることがあります．


In [ ]:
from __future__ import annotations

import urllib.request
from pathlib import Path

import cv2
import gradio as gr
import numpy as np
import torch
from PIL import Image, ImageDraw, ImageFont
from tqdm.auto import tqdm
from transformers import pipeline

# ------------------------------------------------------------
# 定数・サンプル画像 URL・モデル ID
# ------------------------------------------------------------
SAMPLE_DIR = Path("samples_depth_anything")
FONT_DIR = Path("fonts")
FONT_PATH = FONT_DIR / "NotoSansJP-VF.ttf"
FONT_URL = (
    "https://raw.githubusercontent.com/googlefonts/noto-cjk/main/"
    "Sans/Variable/TTF/Subset/NotoSansJP-VF.ttf"
)

MODEL_IDS: dict[str, str] = {
    "small": "depth-anything/Depth-Anything-V2-Small-hf",
    "base": "depth-anything/Depth-Anything-V2-Base-hf",
    "large": "depth-anything/Depth-Anything-V2-Large-hf",
}

# 公開画像（Pexels）．奥行きが分かりやすい題材．
SAMPLE_IMAGE_SOURCES: list[tuple[str, str, str]] = [
    (
        "street.jpg",
        "https://images.pexels.com/photos/1105666/pexels-photo-1105666.jpeg?auto=compress&cs=tinysrgb&w=800",
        "街並み",
    ),
    (
        "room.jpg",
        "https://images.pexels.com/photos/1571460/pexels-photo-1571460.jpeg?auto=compress&cs=tinysrgb&w=800",
        "室内",
    ),
    (
        "mountain.jpg",
        "https://images.pexels.com/photos/417074/pexels-photo-417074.jpeg?auto=compress&cs=tinysrgb&w=800",
        "山・風景",
    ),
    (
        "bicycle.jpg",
        "https://images.pexels.com/photos/100582/pexels-photo-100582.jpeg?auto=compress&cs=tinysrgb&w=800",
        "自転車",
    ),
    (
        "person.jpg",
        "https://images.pexels.com/photos/1239291/pexels-photo-1239291.jpeg?auto=compress&cs=tinysrgb&w=800",
        "人物",
    ),
    (
        "bridge.jpg",
        "https://images.pexels.com/photos/460672/pexels-photo-460672.jpeg?auto=compress&cs=tinysrgb&w=800",
        "橋・遠景",
    ),
]

USER_AGENT = (
    "Mozilla/5.0 (compatible; OpenCampusDemo/1.0; "
    "+https://github.com/yryo1005/OpenCampus_Demo)"
)


def resolve_device() -> str:
    """利用可能な推論デバイスを返す．

    Returns:
        str: "cuda" または "cpu"
    """
    if torch.cuda.is_available():
        name = torch.cuda.get_device_name(0)
        mem_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
        print(f"GPU: {name} ({mem_gb:.1f} GB)")
        return "cuda"
    print("GPU が見つかりません．CPU で実行します（かなり時間がかかります）．")
    return "cpu"


def download_bytes(url: str, save_path: Path) -> Path:
    """URL からバイナリを取得して保存する（既存ならスキップ）．

    Args:
        url (str): ダウンロード元 URL
        save_path (Path): 保存先パス

    Returns:
        Path: 保存したファイルのパス
    """
    if save_path.exists() and save_path.stat().st_size > 0:
        return save_path
    save_path.parent.mkdir(parents=True, exist_ok=True)
    req = urllib.request.Request(url, headers={"User-Agent": USER_AGENT})
    with urllib.request.urlopen(req, timeout=300) as response:
        save_path.write_bytes(response.read())
    return save_path


def download_image(url: str, save_path: Path, max_side: int = 1280) -> Path:
    """URL から画像を取得し，長辺を制限して保存する．

    Args:
        url (str): ダウンロード元 URL
        save_path (Path): 保存先パス
        max_side (int): 長辺の上限ピクセル（既定 1280）

    Returns:
        Path: 保存したファイルのパス
    """
    if save_path.exists() and save_path.stat().st_size > 0:
        return save_path
    save_path.parent.mkdir(parents=True, exist_ok=True)
    req = urllib.request.Request(url, headers={"User-Agent": USER_AGENT})
    with urllib.request.urlopen(req, timeout=60) as response:
        raw = response.read()
    arr = np.frombuffer(raw, dtype=np.uint8)
    bgr = cv2.imdecode(arr, cv2.IMREAD_COLOR)
    if bgr is None:
        raise RuntimeError(f"画像のデコードに失敗しました: {url}")
    h, w = bgr.shape[:2]
    long_side = max(h, w)
    if long_side > max_side:
        scale = max_side / float(long_side)
        bgr = cv2.resize(
            bgr,
            (int(w * scale), int(h * scale)),
            interpolation=cv2.INTER_AREA,
        )
    ok, encoded = cv2.imencode(".jpg", bgr, [int(cv2.IMWRITE_JPEG_QUALITY), 90])
    if not ok:
        raise RuntimeError(f"画像のエンコードに失敗しました: {save_path}")
    save_path.write_bytes(encoded.tobytes())
    return save_path


def download_font(url: str, save_path: Path) -> Path:
    """日本語表示用フォントをダウンロードする（既存ならスキップ）．

    Args:
        url (str): フォントの URL
        save_path (Path): 保存先パス

    Returns:
        Path: 保存したフォントのパス
    """
    return download_bytes(url, save_path)


def prepare_sample_images(
    sources: list[tuple[str, str, str]],
    sample_dir: Path,
) -> list[tuple[str, Path]]:
    """サンプル画像をダウンロードし，ラベルとパスの一覧を返す．

    Args:
        sources (list[tuple[str, str, str]]): (ファイル名, URL, 表示ラベル) のリスト
        sample_dir (Path): 保存先ディレクトリ

    Returns:
        list[tuple[str, Path]]: (表示ラベル, ローカルパス) のリスト
    """
    prepared: list[tuple[str, Path]] = []
    for filename, url, label in tqdm(sources, desc="サンプル画像DL", leave=False):
        path = download_image(url, sample_dir / filename)
        print(f"  {label}: {path} ({path.stat().st_size} bytes)")
        prepared.append((label, path))
    return prepared


def resolve_colormap(name: str) -> int:
    """カラーマップ名を OpenCV の定数に変換する．

    Args:
        name (str): 例 "INFERNO", "TURBO"

    Returns:
        int: cv2.COLORMAP_* 定数
    """
    key = f"COLORMAP_{name.strip().upper()}"
    if not hasattr(cv2, key):
        print(f"未知のカラーマップ '{name}' → INFERNO を使用します．")
        return cv2.COLORMAP_INFERNO
    return int(getattr(cv2, key))


def load_depth_pipeline(model_size: str, device: str):
    """Depth Anything V2 の depth-estimation パイプラインを構築する．

    Args:
        model_size (str): "small" / "base" / "large"
        device (str): "cuda" または "cpu"

    Returns:
        transformers.pipelines.Pipeline: depth-estimation パイプライン
    """
    size_key = model_size.strip().lower()
    if size_key not in MODEL_IDS:
        raise ValueError(f"未知の MODEL_SIZE: {model_size}（small/base/large）")

    model_id = MODEL_IDS[size_key]
    device_index = 0 if device == "cuda" else -1
    print(f"モデルを読み込み中: {model_id} (device={device})")
    depth_pipe = pipeline(
        task="depth-estimation",
        model=model_id,
        device=device_index,
    )
    return depth_pipe


def to_rgb_uint8(image) -> np.ndarray | None:
    """Gradio / PIL / ndarray 入力を RGB uint8 (H, W, 3) に揃える．

    Args:
        image: Gradio Image の入力（None / PIL.Image / np.ndarray）

    Returns:
        np.ndarray | None: RGB 画像．入力が無い場合は None
    """
    if image is None:
        return None
    if isinstance(image, Image.Image):
        return np.asarray(image.convert("RGB"))
    arr = np.asarray(image)
    if arr.ndim == 2:
        return cv2.cvtColor(arr.astype(np.uint8), cv2.COLOR_GRAY2RGB)
    if arr.shape[2] == 4:
        return arr[:, :, :3].astype(np.uint8)
    return arr.astype(np.uint8)


def resize_long_side(rgb: np.ndarray, max_side: int) -> np.ndarray:
    """長辺が max_side を超える場合に縮小する．

    Args:
        rgb (np.ndarray): RGB 画像，形状 (H, W, 3)
        max_side (int): 長辺の上限

    Returns:
        np.ndarray: 縮小後（またはそのまま）の RGB 画像，形状 (H', W', 3)
    """
    h, w = rgb.shape[:2]
    long_side = max(h, w)
    if long_side <= max_side:
        return rgb
    scale = max_side / float(long_side)
    return cv2.resize(
        rgb,
        (int(w * scale), int(h * scale)),
        interpolation=cv2.INTER_AREA,
    )


def prepare_input_image(image, mirror: bool) -> np.ndarray | None:
    """入力画像を RGB 化し，必要なら左右反転・長辺縮小する．

    Args:
        image: Gradio Image 入力
        mirror (bool): True なら水平フリップ

    Returns:
        np.ndarray | None: 前処理後 RGB，形状 (H, W, 3)．入力無しは None
    """
    rgb = to_rgb_uint8(image)
    if rgb is None:
        return None
    if mirror:
        rgb = cv2.flip(rgb, 1)
    return resize_long_side(rgb, int(MAX_IMAGE_SIDE))


def depth_to_colormap(depth_gray: np.ndarray, colormap: int) -> np.ndarray:
    """グレースケール深度を疑似カラー画像（RGB）に変換する．

    Args:
        depth_gray (np.ndarray): 深度のグレースケール，形状 (H, W) または (H, W, 1)
        colormap (int): cv2.COLORMAP_*

    Returns:
        np.ndarray: カラー深度マップ，形状 (H, W, 3)，dtype=uint8，RGB
    """
    gray = np.asarray(depth_gray)
    if gray.ndim == 3:
        gray = gray[:, :, 0]
    gray = gray.astype(np.float32)
    d_min = float(np.min(gray))
    d_max = float(np.max(gray))
    if d_max - d_min < 1e-6:
        norm = np.zeros_like(gray, dtype=np.uint8)
    else:
        norm = ((gray - d_min) / (d_max - d_min) * 255.0).astype(np.uint8)
    bgr = cv2.applyColorMap(norm, colormap)
    return cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)


def make_side_by_side(left_rgb: np.ndarray, right_rgb: np.ndarray) -> np.ndarray:
    """原画像と深度マップを横並びにする．

    Args:
        left_rgb (np.ndarray): 左側画像，形状 (H, W, 3)
        right_rgb (np.ndarray): 右側画像，形状 (H, W, 3)

    Returns:
        np.ndarray: 横連結画像，形状 (H, W*2, 3)
    """
    h = min(left_rgb.shape[0], right_rgb.shape[0])
    w = min(left_rgb.shape[1], right_rgb.shape[1])
    left = cv2.resize(left_rgb, (w, h), interpolation=cv2.INTER_AREA)
    right = cv2.resize(right_rgb, (w, h), interpolation=cv2.INTER_AREA)
    return np.concatenate([left, right], axis=1)


def draw_status_label(rgb: np.ndarray, text: str) -> np.ndarray:
    """画像左上に短い説明テキストを描画する．

    Args:
        rgb (np.ndarray): RGB 画像，形状 (H, W, 3)
        text (str): 描画する文字列

    Returns:
        np.ndarray: 描画後の RGB 画像，形状 (H, W, 3)
    """
    out = rgb.copy()
    pil = Image.fromarray(out)
    draw = ImageDraw.Draw(pil)
    try:
        font = ImageFont.truetype(str(FONT_PATH), size=22)
    except OSError:
        font = ImageFont.load_default()
    draw.rectangle((8, 8, 8 + 18 * len(text) + 16, 42), fill=(0, 0, 0, 180))
    draw.text((16, 12), text, fill=(255, 255, 255), font=font)
    return np.asarray(pil)


def format_depth_summary(depth_gray: np.ndarray, h: int, w: int) -> str:
    """深度推定結果の説明文を作る．

    Args:
        depth_gray (np.ndarray): グレースケール深度，形状 (H, W)
        h (int): 入力画像の高さ
        w (int): 入力画像の幅

    Returns:
        str: 高校生向けの短い説明
    """
    gray = np.asarray(depth_gray, dtype=np.float32)
    if gray.ndim == 3:
        gray = gray[:, :, 0]
    # pipeline の depth は近いほど明るいことが多い
    bright = float(np.percentile(gray, 90))
    dark = float(np.percentile(gray, 10))
    mean_v = float(np.mean(gray))
    return (
        f"画像サイズ: {w}×{h}\n"
        f"モデル: Depth Anything V2 ({MODEL_SIZE})\n"
        f"相対深度の目安（明るい＝手前寄り）\n"
        f"  手前側の明るさ(90%点): {bright:.1f}\n"
        f"  奥側の明るさ(10%点): {dark:.1f}\n"
        f"  平均: {mean_v:.1f}\n"
        "※ メートル単位の絶対距離ではなく，「手前／奥」の相対的な並びです．"
    )


def estimate_depth(
    image,
    mirror: bool,
    colormap_name: str,
) -> tuple[np.ndarray | None, np.ndarray | None, str]:
    """入力画像から深度マップと横並び比較画像を生成する．

    Args:
        image: Gradio Image 入力（カメラ / アップロード / サンプル）
        mirror (bool): 左右反転するか
        colormap_name (str): カラーマップ名（例: INFERNO）

    Returns:
        tuple:
            - 深度カラーマップ RGB (H, W, 3) または None
            - 原画像|深度の横並び RGB (H, 2W, 3) または None
            - 説明文 (str)
    """
    rgb = prepare_input_image(image, mirror=bool(mirror))
    if rgb is None:
        return None, None, "画像をカメラで撮影するか，サンプル／アップロードしてください．"

    pil_image = Image.fromarray(rgb)
    result = depth_pipe(pil_image)
    depth_img = result["depth"]
    if isinstance(depth_img, Image.Image):
        depth_gray = np.asarray(depth_img)
    else:
        depth_gray = np.asarray(depth_img)

    # 入力サイズに合わせて揃える
    if depth_gray.shape[:2] != rgb.shape[:2]:
        depth_gray = cv2.resize(
            depth_gray,
            (rgb.shape[1], rgb.shape[0]),
            interpolation=cv2.INTER_CUBIC,
        )

    cmap = resolve_colormap(colormap_name)
    depth_color = depth_to_colormap(depth_gray, cmap)
    depth_color = draw_status_label(depth_color, "近い ← 明るい色 / 遠い ← 暗い色")
    side = make_side_by_side(rgb, depth_color)
    summary = format_depth_summary(depth_gray, rgb.shape[0], rgb.shape[1])
    return depth_color, side, summary


def build_demo(sample_items: list[tuple[str, Path]]) -> gr.Blocks:
    """カメラ入力・Examples・結果表示を配置した Gradio UI を構築する．

    Args:
        sample_items (list[tuple[str, Path]]): (ラベル, 画像パス)

    Returns:
        gr.Blocks: Gradio デモ
    """
    example_paths = [str(path) for _, path in sample_items]

    with gr.Blocks(title="深度推定デモ（Depth Anything）") as demo:
        gr.Markdown(
            "## Depth Anything で「手前と奥」を色で見る\n"
            "カメラで撮影するか，下のサンプルをクリックしてください．\n"
            "明るい色ほど手前，暗い色ほど奥（相対深度）です．"
        )

        with gr.Row():
            with gr.Column(scale=1):
                image_in = gr.Image(
                    label="入力（カメラ / アップロード）",
                    type="numpy",
                    sources=["webcam", "upload"],
                )
                colormap = gr.Dropdown(
                    choices=["INFERNO", "MAGMA", "TURBO", "PLASMA", "VIRIDIS"],
                    value=str(COLORMAP_NAME).upper(),
                    label="色の付け方",
                )
                mirror = gr.Checkbox(
                    value=bool(MIRROR_WEBCAM),
                    label="左右反転（ミラー）",
                )
                btn_run = gr.Button("深度推定を実行", variant="primary")

            with gr.Column(scale=1):
                image_depth = gr.Image(label="深度マップ", type="numpy")
                image_side = gr.Image(label="原画像 | 深度マップ", type="numpy")
                text_out = gr.Textbox(label="説明", lines=6)

        gr.Examples(
            examples=example_paths,
            inputs=[image_in],
            label="サンプル画像（撮影しなくても試せます）",
        )

        inputs = [image_in, mirror, colormap]
        outputs = [image_depth, image_side, text_out]
        btn_run.click(fn=estimate_depth, inputs=inputs, outputs=outputs)

    return demo

# ------------------------------------------------------------
# 初期化
# ------------------------------------------------------------
DEVICE = resolve_device()
download_font(FONT_URL, FONT_PATH)
sample_items = prepare_sample_images(SAMPLE_IMAGE_SOURCES, SAMPLE_DIR)
depth_pipe = load_depth_pipeline(MODEL_SIZE, DEVICE)
print("準備完了．次のセルで Gradio を起動してください．")


## 3. Gradio の実行

UI が起動したら，サンプル画像をクリックするか，カメラで撮影して「深度推定を実行」を押してください．


In [ ]:
demo = build_demo(sample_items)
demo.launch(share=True, debug=False)
